# 01 — Get the data

Stage 1 of 4. Downloads the **horses** and **races** files for **2010–2020** from the Kaggle dataset
[hwaitt/horse-racing](https://www.kaggle.com/datasets/hwaitt/horse-racing) and saves them, unchanged, to `data/raw/`.

**Needs a free Kaggle account and API token.** On [kaggle.com/settings/api](https://www.kaggle.com/settings/api) click *Generate New Token*, then either
set the environment variable `KAGGLE_API_TOKEN` to it, or save it in the file `~/.kaggle/access_token` (in your home folder, **outside** this repository).
Never put the token in the code or commit it.

Alternative without a token: download the files by hand from the dataset page and put them in `data/raw/`; this notebook then skips the download and only checks them.

**Input:** none  
**Output:** `data/raw/horses_<year>.csv`, `data/raw/races_<year>.csv` for 2010–2020, and `data/raw/dataset-metadata.json` (licence and source information)

In [2]:
import sys
from pathlib import Path

# Make code/helpers.py importable whether the notebook is run from code/ or the repo root.
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "helpers.py").exists() else Path.cwd() / "code"))

import json

import numpy as np
import pandas as pd

import helpers
from helpers import DATA_RAW, DATA_PROCESSED, FIGURES

helpers.set_seed()
helpers.ensure_dirs()

## Settings

In [3]:
DATASET = "hwaitt/horse-racing"
YEARS = helpers.YEARS          # 2010–2020, defined once in helpers.py
KINDS = ("horses", "races")

wanted = [f"{kind}_{year}.csv" for kind in KINDS for year in YEARS]
missing = [name for name in wanted if not (DATA_RAW / name).exists()]
print(f"{len(wanted) - len(missing)} of {len(wanted)} files already in data/raw/; {len(missing)} to download")

22 of 22 files already in data/raw/; 0 to download


## Download the missing files from Kaggle

Skipped when every file is already in `data/raw/`. Otherwise this needs the Kaggle token (see the top of the notebook).
It also saves Kaggle's metadata for the dataset — title, owner and **licence** — to `data/raw/dataset-metadata.json` (Requirement 4.15).

In [4]:
meta_file = DATA_RAW / "dataset-metadata.json"
if missing or not meta_file.exists():
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    for name in missing:
        api.dataset_download_file(DATASET, name, path=DATA_RAW, quiet=True, unzip=True)
        assert (DATA_RAW / name).exists(), f"download failed: {name}"
        print(f"downloaded: {name}")

    if not meta_file.exists():
        api.dataset_metadata(DATASET, path=DATA_RAW)
else:
    print("Nothing to download.")

In [5]:
meta_path = DATA_RAW / "dataset-metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    if isinstance(meta, str):  # some versions of the kaggle package store the JSON as a string
        meta = json.loads(meta)
    print("Title:  ", meta.get("title"))
    print("Licence:", meta.get("licenses"))
else:
    print("No metadata file: record the licence from the Kaggle page in data/README.md by hand.")

Title:   None
Licence: None


## Check what is in `data/raw/`

GitHub rejects files over 100 MB, and its website upload accepts at most 25 MB per file. Files above the limit must stay out of the repository; this notebook then serves as the instructions for obtaining them (Requirement 4.15).

In [6]:
summary = pd.DataFrame(
    [
        {
            "file": name,
            "size_mb": round((DATA_RAW / name).stat().st_size / 1e6, 1),
            "rows": sum(1 for _ in open(DATA_RAW / name, encoding="utf-8", errors="replace")) - 1,
        }
        for name in wanted
    ]
)
summary["under_github_limit"] = summary["size_mb"] < 100
print(f"Total: {summary['size_mb'].sum():.1f} MB, {summary['rows'].sum():,} rows")
summary

Total: 335.6 MB, 1,968,429 rows


,file,size_mb,rows,under_github_limit
0,horses_2010.csv,26.5,157181,True
1,horses_2011.csv,28.2,167448,True
2,horses_2012.csv,27.5,163080,True
3,horses_2013.csv,27.8,164852,True
4,horses_2014.csv,27.2,160577,True
5,horses_2015.csv,26.5,156753,True
6,horses_2016.csv,26.7,157328,True
7,horses_2017.csv,28.3,166865,True
8,horses_2018.csv,29.0,171195,True
9,horses_2019.csv,29.2,171849,True


In [7]:
# A first look at one year of each kind
for kind in KINDS:
    df = pd.read_csv(DATA_RAW / f"{kind}_{YEARS[0]}.csv", nrows=5)
    print(f"{kind}_{YEARS[0]}.csv — columns: {list(df.columns)}")
    display(df)

horses_2010.csv — columns: ['rid', 'horseName', 'age', 'saddle', 'decimalPrice', 'isFav', 'trainerName', 'jockeyName', 'position', 'positionL', 'dist', 'weightSt', 'weightLb', 'overWeight', 'outHandicap', 'headGear', 'RPR', 'TR', 'OR', 'father', 'mother', 'gfather', 'runners', 'margin', 'weight', 'res_win', 'res_place']


,rid,horseName,age,saddle,decimalPrice,isFav,trainerName,jockeyName,position,positionL,...,TR,OR,father,mother,gfather,runners,margin,weight,res_win,res_place
0,272760,Tetlami,4.0,3.0,0.200000,1,Nicky Henderson,Barry Geraghty,1,NaN,...,NaN,NaN,Daylami,Tetou,Peintre Celebre,15,1.208562,70,1.0,1
1,272760,Romulus D'Artaix,4.0,14.0,0.142857,0,Alan King,Robert Thornton,2,4,...,NaN,NaN,Sassanian,Kadisha,Always Fair,15,1.208562,68,0.0,1
2,272760,Marleno,4.0,9.0,0.024390,0,Milton Harris,Warren Marston,3,nk,...,NaN,NaN,Lecroix,Mondalita,Alkalde,15,1.208562,68,0.0,1
3,272760,My Brother Sylvest,4.0,10.0,0.076923,0,John Flint,Rhys Flint,4,5,...,NaN,NaN,Bach,Senna Da Silva,Prince Of Birds,15,1.208562,67,0.0,0
4,272760,Karasenir,4.0,8.0,0.076923,0,Philip Hobbs,Richard Johnson,5,.75,...,NaN,NaN,Sendawar,Karasta,Lake Coniston,15,1.208562,68,0.0,0


races_2010.csv — columns: ['rid', 'course', 'time', 'date', 'title', 'rclass', 'band', 'ages', 'distance', 'condition', 'hurdles', 'prizes', 'winningTime', 'prize', 'metric', 'countryCode', 'ncond', 'class']


,rid,course,time,date,title,rclass,band,ages,distance,condition,hurdles,prizes,winningTime,prize,metric,countryCode,ncond,class
0,272760,Cheltenham,03:50,10/01/01,"EBF ""High Sheriff of Gloucestershire's"" ""Junio...",Class 1,NaN,4yo,1m6f,Soft,NaN,"[10115.0, 3787.6, 1892.1, 945.2]",222.70,16740,2815,GB,5,1
1,280996,Cheltenham,01:40,10/01/01,Dipper Novices' Chase Grade 2,Class 1,NaN,5yo+,2m5f,Soft,17 fences,"[18243.2, 6844.8]",357.30,25088,4223,GB,5,1
2,286408,Southwell (AW),01:35,10/01/01,Betdaq The Betting Exchange Handicap,Class 5,0-75,4yo+,1m6f,Standard To Slow,NaN,"[2331.36, 693.72, 346.68, 173.16]",195.35,3545,2815,GB,13,5
3,288208,Tramore (IRE),02:30,10/01/01,New Years Day at Tramore Racecourse 6YO Maiden...,NaN,NaN,6yo,2m,Heavy,10 hurdles,"[5182.5, 1207.5, 532.5, 307.5]",268.00,7230,3218,IE,9,0
4,297757,Southwell (AW),12:25,10/01/01,Happy New Year Apprentice Median Auction Maide...,Class 6,NaN,4-6yo,1m3f,Standard To Slow,NaN,"[1774.24, 523.9, 262.08]",154.92,2560,2212,GB,13,6
